# Modelo 9 — Random Survival Forest

Mesmo painel e a mesma pergunta operacional do Modelo 5 (ZM01 em 7 dias), com outro alvo: tempo até a próxima corretiva e censura quando ela ainda não ocorreu. Na hora de alertar, usamos P(T ≤ 7).

## 1. Dados

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv

raiz = Path.cwd()
if not (raiz / 'models' / 'modelo-5' / 'painel.parquet').exists():
    for pasta in Path.cwd().parents:
        candidato = pasta / 'models' / 'modelo-5' / 'painel.parquet'
        if candidato.exists():
            raiz = pasta
            break

painel = pd.read_parquet(raiz / 'models' / 'modelo-5' / 'painel.parquet')
painel['Data'] = pd.to_datetime(painel['Data'])
painel = painel.sort_values(['id_porta', 'Data']).reset_index(drop=True)
print(painel.shape, painel['Data'].min().date(), '->', painel['Data'].max().date())

## 2. Features

In [ ]:
PROIBIDAS = {'n_falhas_7d_futuro'}
EPS = 1.0

for prefixo in ('n_total', 'n_LSD', 'n_tecnicos', 'n_long30', 'n_FO', 'n_FC', 'n_RCE'):
    painel[f'rampa_{prefixo}'] = painel[f'{prefixo}_7d'] - (painel[f'{prefixo}_14d'] - painel[f'{prefixo}_7d'])

painel['razao_7_30'] = painel['n_total_7d'] / (painel['n_total_30d'] / 4 + EPS)
painel['razao_lsd_7_30'] = painel['n_LSD_7d'] / (painel['n_LSD_30d'] / 4 + EPS)
painel['frac_tec_7'] = painel['n_tecnicos_7d'] / (painel['n_total_7d'] + EPS)
painel['frac_lsd_7'] = painel['n_LSD_7d'] / (painel['n_tecnicos_7d'] + EPS)
painel['frac_long_7'] = painel['n_long30_7d'] / (painel['n_total_7d'] + EPS)
painel['lsd_x_dur'] = painel['n_LSD_7d'] * np.log1p(painel['dur_max_7d'])
painel['frescor_total'] = painel['n_total_decaido_h3'] / (painel['n_total_decaido_h14'] + EPS)

for origem, dest in (
    ('dias_desde_zm01', 'rec_zm01'),
    ('dias_desde_zm02', 'rec_zm02'),
    ('dias_desde_zm03', 'rec_zm03'),
):
    dias = painel[origem].replace(9999, np.nan).clip(upper=365).fillna(365)
    painel[dest] = np.exp(-dias / 30.0)

painel['zm01_concentracao'] = painel['zm01_30d'] / (painel['zm01_90d'] / 3 + EPS)
dias_z = painel['dias_desde_zm01'].replace(9999, 365).clip(0, 365)
painel['razao_atraso'] = dias_z / (90.0 / painel['zm01_90d'].clip(lower=0.5))

id_cols = {
    'id_porta', 'Data', 'est', 'via', 'id_plat', 'y', 'cobertura_30d',
    'mes', 'dia_semana', 'fim_de_semana', 'num_porta',
}
candidatas = [
    c for c in painel.columns
    if c not in PROIBIDAS and c not in id_cols
    and not c.startswith('dias_desde_')
    and not c.endswith(('_1d', '_3d'))
    and pd.api.types.is_numeric_dtype(painel[c])
]
print(len(candidatas), 'candidatas')

## 3. Alvo de sobrevivência e split

In [ ]:
FIM_FIT = pd.Timestamp('2025-01-23')
INI_VALID = pd.Timestamp('2025-01-31')
FIM_VALID = pd.Timestamp('2025-04-23')
INI_TESTE = pd.Timestamp('2025-05-01')
HORIZONTE = 7

falhas = painel.loc[painel['dias_desde_zm01'] == 1, ['id_porta', 'Data']].copy()
falhas['falha'] = falhas['Data'] - pd.Timedelta(days=1)
falhas = falhas[['id_porta', 'falha']].drop_duplicates().sort_values(['id_porta', 'falha'])
print('falhas inferidas', len(falhas), 'portas', falhas['id_porta'].nunique())


def rotulo_sobrevivencia(df, fim_obs):
    ev = falhas.loc[falhas['falha'] <= fim_obs, ['id_porta', 'falha']].copy()
    ev['Data'] = ev['falha']
    base = df[['id_porta', 'Data']].copy()
    base['_idx'] = np.arange(len(base))
    m = pd.merge_asof(
        base.sort_values('Data'),
        ev.sort_values('Data'),
        on='Data',
        by='id_porta',
        direction='forward',
        allow_exact_matches=False,
    )
    prox = pd.to_datetime(m['falha'].values)
    t0 = pd.to_datetime(m['Data'].values)
    delta = (prox - t0) / pd.Timedelta(days=1)
    evento = prox.notna() & (prox <= fim_obs) & (delta > 0)
    tempo = np.where(evento, delta, (fim_obs - t0) / pd.Timedelta(days=1))
    tempo = np.clip(tempo, 1, None)
    ordem = m['_idx'].to_numpy()
    ev_out = np.empty(len(df), dtype=bool)
    tm_out = np.empty(len(df), dtype=float)
    ev_out[ordem] = evento
    tm_out[ordem] = tempo
    return pd.Series(ev_out, index=df.index), pd.Series(tm_out, index=df.index)


painel = painel.sort_values(['Data', 'id_porta']).reset_index(drop=True)
treino = painel.loc[painel['Data'] <= FIM_FIT].copy()
valid = painel.loc[painel['Data'].between(INI_VALID, FIM_VALID)].copy()
teste = painel.loc[painel['Data'] >= INI_TESTE].copy()

treino['evento'], treino['tempo'] = rotulo_sobrevivencia(treino, FIM_FIT + pd.Timedelta(days=HORIZONTE))
valid['evento'], valid['tempo'] = rotulo_sobrevivencia(valid, FIM_VALID + pd.Timedelta(days=HORIZONTE))
teste['evento'], teste['tempo'] = rotulo_sobrevivencia(teste, painel['Data'].max())

y7_tr = (treino['evento'] & (treino['tempo'] <= HORIZONTE)).astype(int)
y7_va = (valid['evento'] & (valid['tempo'] <= HORIZONTE)).astype(int)
y7_te = (teste['evento'] & (teste['tempo'] <= HORIZONTE)).astype(int)

print('treino', treino['Data'].min().date(), '->', treino['Data'].max().date(),
      '| eventos', int(treino['evento'].sum()), '| y7', float(y7_tr.mean()))
print('valid ', valid['Data'].min().date(), '->', valid['Data'].max().date(), '| y7', float(y7_va.mean()))
print('teste ', teste['Data'].min().date(), '->', teste['Data'].max().date(), '| y7', float(y7_te.mean()))
print('acordo y do painel no teste', float((y7_te.to_numpy() == teste['y'].to_numpy()).mean()))

## 4. Seleção e matriz

In [ ]:
def ap_univariada(serie, y):
    v = pd.to_numeric(serie, errors='coerce').replace([np.inf, -np.inf], np.nan)
    v = v.fillna(v.median() if v.notna().any() else 0.0)
    return max(average_precision_score(y, v), average_precision_score(y, -v))


base_ap = float(y7_tr.mean())
ranking = {c: ap_univariada(treino[c], y7_tr) for c in candidatas}
ordem = sorted(ranking, key=ranking.get, reverse=True)
feats = [c for c in ordem if ranking[c] >= base_ap * 1.08][:32]
X_corr = treino[feats].replace([np.inf, -np.inf], np.nan).fillna(0)
corr = X_corr.corr().abs()
kept = []
for c in feats:
    if X_corr[c].std() < 1e-9:
        continue
    if all(corr.loc[c, k] < 0.94 for k in kept):
        kept.append(c)
feats = kept
assert 'y' not in feats and 'n_falhas_7d_futuro' not in feats
print(len(feats), 'features')
print(*[f'{c}:{ranking[c]:.3f}' for c in feats[:10]], sep=' | ')


def montar_X(df):
    return df[feats].replace([np.inf, -np.inf], np.nan).fillna(0).astype(float)


X_tr, X_va, X_te = montar_X(treino), montar_X(valid), montar_X(teste)
ys_tr = Surv.from_arrays(treino['evento'].to_numpy(), treino['tempo'].to_numpy())
ys_va = Surv.from_arrays(valid['evento'].to_numpy(), valid['tempo'].to_numpy())
print(X_tr.shape, X_va.shape, X_te.shape)

## 5. Random Survival Forest

In [ ]:
rng = np.random.RandomState(42)
houve = np.flatnonzero(treino['evento'].to_numpy())
cens = np.flatnonzero(~treino['evento'].to_numpy())
n_ev = min(len(houve), 16000)
n_cen = min(len(cens), max(32000, n_ev * 2))
idx = np.concatenate([
    rng.choice(houve, n_ev, replace=False),
    rng.choice(cens, n_cen, replace=False),
])
print('amostra de treino', len(idx))

rsf = RandomSurvivalForest(
    n_estimators=120,
    min_samples_leaf=40,
    max_depth=12,
    max_features='sqrt',
    n_jobs=-1,
    random_state=42,
)
rsf.fit(X_tr.iloc[idx], ys_tr[idx])


def risco_7d(modelo, X):
    curvas = modelo.predict_survival_function(X, return_array=False)
    return np.array([1.0 - float(fn(HORIZONTE)) for fn in curvas])


score_va = risco_7d(rsf, X_va)
score_te = risco_7d(rsf, X_te)

grid = np.unique(np.quantile(score_va, np.linspace(0.45, 0.985, 150)))
limiar = max((f1_score(y7_va, score_va >= t, zero_division=0), float(t)) for t in grid)[1]
pred_te = (score_te >= limiar).astype(int)

print('limiar', round(limiar, 4), '| F1 val', round(f1_score(y7_va, score_va >= limiar, zero_division=0), 3))
print('PR-AUC teste', round(average_precision_score(y7_te, score_te), 4),
      '| baseline', round(float(y7_te.mean()), 4))
print('classe 1  P:', round(precision_score(y7_te, pred_te, zero_division=0), 3),
      'R:', round(recall_score(y7_te, pred_te, zero_division=0), 3),
      'F1:', round(f1_score(y7_te, pred_te, zero_division=0), 3))
print(classification_report(y7_te, pred_te, digits=3, zero_division=0))
print('Modelo 5 (mesmo teste): P 0.159  R 0.553  F1 0.245  PR-AUC 0.186')

## 6. Avaliação por evento

In [ ]:
def n_inicios(df, col):
    s = df.sort_values(['id_porta', 'Data'])
    flag = s[col].astype(bool)
    prev = flag.groupby(s['id_porta']).shift(1).fillna(False).astype(bool)
    return int((flag & ~prev).sum())


def eventos_pegos(df, y_col, pred_col):
    s = df.sort_values(['id_porta', 'Data']).copy()
    y = s[y_col].astype(bool)
    novo = y & ~y.groupby(s['id_porta']).shift(1).fillna(False).astype(bool)
    s['_eid'] = novo.groupby(s['id_porta']).cumsum()
    falhas = s[y]
    if falhas.empty:
        return 0, 0
    grupos = falhas.groupby(['id_porta', '_eid'], sort=False)
    return int(grupos[pred_col].max().sum()), int(grupos.ngroups)


avaliar = teste.copy()
avaliar['pred'] = pred_te
avaliar['y7'] = y7_te.to_numpy()
pegos, n_eventos = eventos_pegos(avaliar, 'y7', 'pred')
n_avisos = n_inicios(avaliar, 'pred')
avisos_ok, _ = eventos_pegos(avaliar, 'pred', 'y7')
print('eventos', n_eventos, 'pegos', pegos, 'recall evento', round(pegos / n_eventos, 3) if n_eventos else 0)
print('avisos', n_avisos, 'acertaram', avisos_ok, 'precisão evento', round(avisos_ok / n_avisos, 3) if n_avisos else 0)

## 7. Curva Precision-Recall

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4))
p, r, _ = precision_recall_curve(y7_te, score_te)
ax.plot(r, p, label=f'RSF PR-AUC {average_precision_score(y7_te, score_te):.3f}')
ax.axhline(y7_te.mean(), color='gray', linestyle='--', label=f'baseline {y7_te.mean():.3f}')
ax.set_xlabel('Recall')
ax.set_ylabel('Precisão')
ax.legend()
fig.tight_layout()
plt.show()